# Rock Paper Scissors

In this tutorial you will learn how to build a Xian smart contract for the classic game Rock, Paper, Scissors. Through this contract, two players can play Rock, Paper, Scissors over a deterministic smart contract runtime.


This is an advanced example. Please make sure you have gone over the previous examples before this one so you have a better understanding of contracting as a whole. 

### Important Disclaimer:
This is an example smart contract. It is not production ready. It needs more tests. It also does not address certain timing edge cases. For example a user can just not reveal their choice and stall a game forever. The solution is left as an exercise for you. It is important when building a smart contract that you think of all edge cases and test heavily before deploying it to the blockchain. 

First let's import some things we will need later.

In [ ]:
from contracting.stdlib.bridge.hashing import sha3
from contracting.client import ContractingClient
import secrets

Below is the contract source for Rock, Paper, Scissors. In modern notebooks it is more reliable to submit explicit source strings than to depend on notebook-cell source introspection.


In [ ]:
rps_contract = """
next_game_id = Variable()

game_id_to_password_hash = Hash()

game_id_to_player1_choice_hash = Hash()
game_id_to_player2_choice_hash = Hash()

game_id_to_player1_choice = Hash()
game_id_to_player2_choice = Hash()

@construct
def constructor():
    next_game_id.set(0)


@export
def start_game(password_hash: str, player1_choice_hash: str):
    unique_game_id = next_game_id.get()
    assert get_game_state(unique_game_id) == "game_doesnt_exist", (
        "this is a bug in the contract. new game id already exists"
    )
    next_game_id.set(next_game_id.get() + 1)
    game_id_to_password_hash[str(unique_game_id)] = password_hash
    game_id_to_player1_choice_hash[str(unique_game_id)] = player1_choice_hash
    return unique_game_id


@export
def submit_choice(game_id: int, game_password: str, player2_choice_hash: str):
    assert get_game_state(game_id) == "only_player1_submitted", (
        "submit_choice can only be called if only player 1 has submitted their choice"
    )
    assert hashlib.sha3(game_password) == game_id_to_password_hash[str(game_id)], "Wrong password!"
    game_id_to_player2_choice_hash[str(game_id)] = player2_choice_hash
    assert get_game_state(game_id) == "both_players_submitted", (
        "this is a bug in the contract. after submitting player2 choice both players must have submitted"
    )


@export
def get_game_state(game_id: int):
    if next_game_id.get() <= game_id:
        return "game_doesnt_exist"

    player1_hashed_choice = game_id_to_player1_choice_hash[str(game_id)]
    player2_hashed_choice = game_id_to_player2_choice_hash[str(game_id)]

    if player1_hashed_choice is not None and player2_hashed_choice is None:
        return "only_player1_submitted"

    player1_choice = game_id_to_player1_choice[str(game_id)]
    player2_choice = game_id_to_player2_choice[str(game_id)]

    if (
        player1_hashed_choice is not None
        and player2_hashed_choice is not None
        and player1_choice is None
        and player2_choice is None
    ):
        return "both_players_submitted"

    assert player1_hashed_choice is not None and player2_hashed_choice is not None, (
        "this is a bug in the contract. error code 1"
    )

    if player1_choice is not None and player2_choice is None:
        return "only_player1_revealed"

    if player1_choice is None and player2_choice is not None:
        return "only_player2_revealed"

    assert player1_choice is not None and player2_choice is not None, (
        "this is a bug in the contract. error code 2"
    )

    if player1_choice == player2_choice:
        return "tie"

    if beats(player1_choice, player2_choice):
        return "player1_wins"

    if beats(player2_choice, player1_choice):
        return "player2_wins"


@export
def is_valid_choice(choice: str):
    return choice in ["rock", "paper", "scissors"]


@export
def beats(choice1: str, choice2: str):
    assert is_valid_choice(choice1), "choice1 must be a valid choice"
    assert is_valid_choice(choice2), "choice2 must be a valid choice"

    if choice1 == "rock" and choice2 == "scissors":
        return True
    if choice1 == "paper" and choice2 == "rock":
        return True
    if choice1 == "scissors" and choice2 == "paper":
        return True
    return False


@export
def reveal(game_id: int, choice: str, choice_salt: str, is_player1: bool):
    if is_player1:
        assert get_game_state(game_id) in ["both_players_submitted", "only_player2_revealed"], (
            "reveal can only be called by player1 if no player or only player2 has revealed"
        )
    else:
        assert get_game_state(game_id) in ["both_players_submitted", "only_player1_revealed"], (
            "reveal can only be called by player2 if no player or only player1 has revealed"
        )

    assert is_valid_choice(choice), "choice must be rock, paper or scissors"

    salted_choice = choice + choice_salt
    hashed_choice = hashlib.sha3(salted_choice)

    if is_player1:
        assert game_id_to_player1_choice_hash[str(game_id)] == hashed_choice, (
            "Player 1 has revealed a choice different from what they submitted"
        )
    else:
        assert game_id_to_player2_choice_hash[str(game_id)] == hashed_choice, (
            "Player 2 has revealed a choice different from what they submitted"
        )

    if is_player1:
        game_id_to_player1_choice[str(game_id)] = choice
    else:
        game_id_to_player2_choice[str(game_id)] = choice
"""


Welcome back! Lets get this contract into the blockchain. To interact with the blockchain we need a client. 

In [ ]:
client = ContractingClient(signer='ren')

Get rid of all state so we have a blank slate. That keeps the notebook rerunnable.


In [ ]:
client.flush()

Now we submit the contract source to the runtime. The contract name must be lowercase and start with `con_`.


In [ ]:
client.submit(rps_contract, name="con_rps")


Get a handle for the contract that we can interact with.

In [ ]:
contract = client.get_contract("con_rps")


With the contract in the blockchain we can now play Rock, Paper, Scissors. Our players for this example are Alice and Bob.

Alice (player 1) chooses single use password. Only the person that has the password can join the game and play with Alice. Everything that starts with alice_ is only visible to Alice. 

In [ ]:
alice_game_password = "trollbridge"

Alice hashes the password so she can submit it to the blockchain without sharing the actual password. She does this because everything on the blockchain is public, and she wants only the person she chooses to play the game with to have the password. 

In [ ]:
alice_game_password_hash = sha3(alice_game_password)
alice_game_password_hash

Alice chooses rock.

In [ ]:
alice_choice = "rock"

We can't submit the choice to the blockchain as plain text, because then Bob (player 2) can see it and win by choosing paper. 

In [ ]:
alice_choice_hash = sha3(alice_choice)
alice_choice_hash

The problem with submitting Alices choice like this is the 3 choices will be hashed the same every time. Bob (player 2) can know what each of the hashes for the 3 choices and pick paper to win. To fix this Alice needs to pick a random salt to hash with her choice so that Bob can't guess her choice by looking at the hash.

In [ ]:
alice_choice_salt = secrets.token_hex(32)
alice_choice_salt

Now we can combine alice_choice and alice_choice_salt and hash them together to create something that Bob can't guess Alices choice from. But Alice can later submit her choice and the salt to prove her choice.

In [ ]:
alice_salted_choice = alice_choice + alice_choice_salt
alice_salted_choice

In [ ]:
alice_salted_choice_hash = sha3(alice_salted_choice)
alice_salted_choice_hash

Before a game is started it is in state `"game_doesnt_exist"`

In [ ]:
assert contract.get_game_state(game_id=0) == "game_doesnt_exist"

Now Alice starts a game so she can invite Bob to play.

In [ ]:
alice_game_id = contract.start_game(password_hash=alice_game_password_hash, player1_choice_hash=alice_salted_choice_hash) 
alice_game_id

Alice gets back a game Id.

The game is now in state "only_player1_submitted"

In [ ]:
assert contract.get_game_state(game_id=alice_game_id) == "only_player1_submitted"

Now Alice has to tell Bob the password and the game Id. This could be done over a messenger or built into the frontend of an application. 

Everything that starts with bob_ is only visible to Bob.

In [ ]:
bob_game_password = alice_game_password
bob_game_id = alice_game_id

Now it is Bobs turn.
Bob chooses scissors.

In [ ]:
bob_choice = "scissors"

And now Bob has to salt his choice.

In [ ]:
bob_choice_salt = secrets.token_hex(32)
bob_choice_salt

And now we combine bob_choice and bob_choice_salt together and hash them.

In [ ]:
bob_salted_choice = bob_choice + bob_choice_salt
bob_salted_choice

In [ ]:
bob_salted_choice_hash = sha3(bob_salted_choice)
bob_salted_choice_hash

Bob now needs to submit his choice to the blockchain. Only Bob has the game password so only Bob can join Alices game.

In [ ]:
contract.submit_choice(game_password=bob_game_password, game_id=bob_game_id, player2_choice_hash=bob_salted_choice_hash)

The game is now in state "both_players_submitted"

In [ ]:
assert contract.get_game_state(game_id=bob_game_id) == "both_players_submitted"

Now that both players have submitted their hashed and salted choices, both players can reveal their choices.
The order doesn't matter. Alice goes first in this example.

In [ ]:
contract.reveal(game_id=alice_game_id, choice=alice_choice, choice_salt=alice_choice_salt, is_player1=True)

The game is now in state "only_player1_revealed"

In [ ]:
assert contract.get_game_state(game_id=alice_game_id) == "only_player1_revealed"

Bob goes second

In [ ]:
contract.reveal(game_id=bob_game_id, choice=bob_choice, choice_salt=bob_choice_salt, is_player1=False)

Now we can see who won the game

In [ ]:
contract.get_game_state(game_id=alice_game_id)

As expected Alice, who is player1, wins!

## tests

Test function is_valid_choice

In [ ]:
assert contract.is_valid_choice(choice="rock")
assert contract.is_valid_choice(choice="paper")
assert contract.is_valid_choice(choice="scissors")
assert not contract.is_valid_choice(choice="airplane")

Test function beats

In [ ]:
assert contract.beats(choice1="rock", choice2="scissors")
assert not contract.beats(choice1="rock", choice2="rock")
assert not contract.beats(choice1="rock", choice2="paper")

assert contract.beats(choice1="paper", choice2="rock")
assert not contract.beats(choice1="paper", choice2="paper")
assert not contract.beats(choice1="paper", choice2="scissors")

assert contract.beats(choice1="scissors", choice2="paper")
assert not contract.beats(choice1="scissors", choice2="scissors")
assert not contract.beats(choice1="scissors", choice2="rock")